# 10 Custom Attention: Monitoring, Read-Only Top-K, and KV Layouts

After reading Examples 00, 06, and 07, if you are interested in attention-level inference-time algorithms—such as attention monitoring (measuring where attention mass lands), training-free sparse reads (such as Quest page-level selection), or altering KV storage layouts (sliding window, prompt sink plus recent window, or cross-layer KV sharing)—they share one property: none of them modify q/k/v projections; they only change **how a layer reads KV** or **how KV is stored**. In vLLM, however, attention is a fused kernel and KV is paged and managed by the scheduler; changing either usually requires editing the attention backend or model code.

`@pllm.custom_attention` replaces only the attention read at the specified layers: native projections and KV writes proceed as usual, while your method receives the current layer's (RoPE-applied) `q` and a read-only paged KV view `kv`. You invoke the native kernel via `kv.attend(...)` (optionally with a window or per-head position selection), return the attention output, and can optionally return an extra `aux` tensor for downstream methods to read. Meanwhile, `keep=`, `window=`, `sink=`, and `source=` on the same decorator declare how KV is stored, leaving actual allocation and reclamation to vLLM's own sliding-window, R-SWA, or KV-sharing mechanisms.

Before starting, keep these facts, limitations, and costs in mind:
- Qwen2.5-0.5B was not trained with these layouts. Cross-layer sharing and even-layer sliding windows corrupt its answers here (see Step 3); this example demonstrates wiring, checks, and costs, not generation quality.
- Read-only top-K (selective reading) saves data read volume, not KV VRAM. Here page scoring is written directly in PyTorch, reading every page's keys at every layer and step, which drops 7B throughput to 13% of native: it illustrates semantics, not a fast implementation. Real speedups require maintaining per-page min/max incrementally and writing a custom kernel (see Example 09).
- Score-based physical KV memory eviction (such as H2O or SnapKV) is unsupported in pLLM (`Score-based memory eviction is unsupported`). vLLM assumes each request's KV is a contiguous prefix starting from position 0, deriving positions and block tables from the number of computed tokens with append-only block tables; punching holes in the middle would break those underlying invariants.
- Execution constraints: requires single-GPU (`tensor_parallel_size=1`), the FlashAttention 2 backend with unquantized KV, and prefix caching disabled; static layouts cannot be combined with `PositionState` or `@pllm.custom_forward`.

This example walks through three steps (measured on RTX 4090, vLLM 0.32.0, bf16; Qwen2.5-0.5B-Instruct for behavior, and Qwen2.5-7B-Instruct at the end of Step 3 for KV accounting and throughput):
1. **Monitoring**: read out the fraction of attention mass falling on recent positions without altering model output.
2. **Read-only top-K**: implement Quest-style page selection and compare it against reading only recent positions under the same budget.
3. **KV layouts**: compare how much KV VRAM sliding window, prompt-sink plus window, and cross-layer sharing each save, along with their throughput.

To observe how different read patterns and layouts behave over long contexts, we first prepare a ~1500-token test prompt: bury `"The secret code for the blue door is 7361."` inside a museum passage and ask the model for the code. Let's run vanilla vLLM first as our baseline.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

from collections import defaultdict
from dataclasses import dataclass, replace

import torch
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing
from pllm.benchmark import throughput

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
# 2048-token context; chunked prefill up to 256 tokens/step; 1 GiB pLLM CUDA graph capture budget
# (ExecutionOptions apply only to engines with a workload; the same config also launches vanilla vLLM.)
CONFIG = pllm.ModelConfig(MODEL, max_model_len=2048, max_num_seqs=8, max_num_batched_tokens=256,
                          enable_chunked_prefill=True, gpu_memory_utilization=0.4, seed=0,
                          execution=pllm.ExecutionOptions(cuda_graph_pool_budget_bytes=1 << 30))

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
filler = ("The museum opens at nine in the morning and closes at six in the evening. Visitors can buy tickets at the "
          "front desk or online. The cafe on the second floor serves coffee, tea and sandwiches. ")
passage = filler * 6 + "The secret code for the blue door is 7361. " + filler * 30
question = "What is the secret code for the blue door? Answer with the number only."
prompt = tokenizer.apply_chat_template([{"role": "user", "content": passage + "\n\n" + question}], tokenize=False,
                                       add_generation_prompt=True)
greedy = SamplingParams(max_tokens=12, temperature=0.0)
print("prompt 长度:", len(tokenizer(prompt).input_ids), "token")


def answer(llm) -> str:
    return llm.generate([prompt], greedy, use_tqdm=False)[0].outputs[0].text


with pllm.make_llm(CONFIG) as plain:
    print("原生 vLLM 的回答:", repr(answer(plain)))

prompt 长度: 1500 token
原生 vLLM 的回答: '7361'


Vanilla vLLM answers `'7361'` on this 1500-token prompt. Now let's move to Step 1: **How do we monitor what fraction of a layer's attention falls on recent positions without changing the model's output?**

Here we want to extract an intermediate statistic during attention computation and export it to the host. `kv.attend(q, *, window=0, topk=None)` returns `(output [rows, heads, head_dim], float32 LSE [rows, heads])`; without selection arguments it invokes the native causal kernel, where LSE is the log-sum-exp (the log of the softmax normalization constant).

The `RecentMass` workload below attaches the custom attention method `split` only at layer 12:
- First, call `kv.attend(q)` for the native read to get the normal `out` and full-sequence `lse`;
- Next, call `kv.attend(q, window=64)` to read only the most recent 64 positions (`window` counts backward from the current token inclusive), obtaining the local `near` (local LSE);
- Subtract and exponentiate `(near - lse).exp()` to compute the attention share falling on the most recent 64 positions, average across heads, and return it as the second output (`aux`);
- Return the native `out` as the first value, so model behavior stays unchanged.

Then, `report` decorated with `@pllm.export` reads this `aux` tensor from layer 12 via `pllm.attention(12, "aux")` (inside an export method it receives the sampled rows; passing `"lse"` would read the LSE of that layer's first `kv.attend`), exports it to the host every step, and `on_export` collects it per request (`prompt_index`).

We run two requests together for contrast: the 1500-token museum prompt and the short `"What is 2 + 3?"`.

In [3]:
@dataclass(frozen=True)
class RecentMass(pllm.Workload):
    recent: int = 64

    @pllm.custom_attention(layers=(12,))
    def split(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> tuple[torch.Tensor, torch.Tensor]:
        out, lse = kv.attend(q)                     # Native read, also capturing log-sum-exp
        _, near = kv.attend(q, window=self.recent)  # Read only the most recent `recent` positions
        return out, (near - lse).exp().mean(-1, keepdim=True)  # aux: fraction of attention on recent positions, averaged over heads

    @pllm.export(name="recent_mass", max_rows=8, max_cols=1)
    def report(self, share: torch.Tensor = pllm.attention(12, "aux")) -> torch.Tensor:
        return share

In [4]:
shares: dict[int, list[float]] = defaultdict(list)


@dataclass(frozen=True)
class RecordedMass(RecentMass):
    def on_export(self, event: pllm.io.TraceEvent) -> None:
        for identity, row in zip(event.identities, event.tensor().tolist()):
            shares[identity.prompt_index].append(round(row[0], 3))


short = tokenizer.apply_chat_template([{"role": "user", "content": "What is 2 + 3?"}], tokenize=False,
                                      add_generation_prompt=True)
with pllm.make_llm(CONFIG, workload=RecordedMass()) as llm:
    outputs = llm.generate([prompt, short], greedy, use_tqdm=False)
    print("eager 原因:", repr(llm.diagnostics().eager_reason))
for index, output in enumerate(outputs):
    print(f"prompt {index}: {output.outputs[0].text!r}")
    print("    最近 64 个位置的注意力占比:", shares[index])

eager 原因: ''
prompt 0: '7361'
    最近 64 个位置的注意力占比: [0.633, 0.588, 0.666, 0.757, 0.586]
prompt 1: '2 + 3 equals 5.'
    最近 64 个位置的注意力占比: [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]


Look at the output: `eager_reason` is `''`, confirming this monitoring workload runs inside CUDA graphs. The long prompt (`prompt 0`) still answers `'7361'`; at each generated token, the most recent 64 positions at layer 12 receive `0.586` to `0.757` of the attention mass, with the rest going to earlier positions. The short prompt (`prompt 1`) answers `'2 + 3 equals 5.'` with `1.0` at every step—because its prompt is only 37 tokens, even with the generated tokens the total length stays under 64 tokens. At the end, `check_on_gpu` compares this monitoring workload against the vanilla model's full-vocabulary distribution.

Next, Step 2: **When each layer and head may only read a fixed small budget of positions, how does page-based top-K selection (Quest) compare to simply reading the most recent positions?**

In plain terms, Quest (Tang et al., 2024) works like this: in vLLM, KV is stored in pages (16 tokens per page here). For each page, record the per-dimension minimum and maximum across all keys on that page; given the current query vector $q$, $\sum_d \max(q_d \cdot \text{min}_d, q_d \cdot \text{max}_d)$ upper-bounds the maximum possible value of $q \cdot k$ on that page. Each attention head picks the pages with the highest upper bounds and attends only to those pages.

In the code below:
- `page_scores` computes the per-row, per-head, per-page upper bound directly in PyTorch from `kv.key_cache` and `kv.block_table` (Qwen2.5-0.5B uses GQA with 14 query heads grouped over 2 KV heads).
- `QuestRead` attaches to layers 2 through 23 (layers 0 and 1 read all positions normally). It masks unwritten pages to `-inf`, selects the top `pages` per head, converts them to logical token positions, and passes them to `kv.attend(q, topk=...)` (`topk` accepts shape `[rows, k]` or `[rows, heads, k]`, padded with `-1` where fewer positions exist; budgets may differ across heads and layers).
- As a control, when `recent_only=True`, it calls `kv.attend(q, window=128)` under the same budget (8 pages = 128 tokens) to read only the most recent 128 positions. We also define `NativeRead`, which reads all positions at every layer as both a control here and the baseline for KV layouts in the next step.

The next cell runs `pllm.explain` and contract checks on CPU to inspect read cost estimates, followed by testing the secret-code question on GPU.

In [5]:
def page_scores(q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
    """Quest: upper-bound the maximum q·k on each page using per-dimension min/max of its keys."""
    keys = kv.key_cache[kv.block_table[kv.requests].long()].float()  # [rows, pages, pos_in_page, kv_heads, head_dim]
    low, high = keys.amin(2), keys.amax(2)                           # [rows, pages, kv_heads, head_dim]
    rows, heads, dim = q.shape
    g = q.float().reshape(rows, low.shape[2], heads // low.shape[2], dim)  # GQA: group by KV head
    bound = (torch.einsum("rkgd,rpkd->rkgp", g.clamp(min=0), high)
             + torch.einsum("rkgd,rpkd->rkgp", g.clamp(max=0), low))
    return bound.reshape(rows, heads, -1)                            # [rows, heads, pages]


@dataclass(frozen=True)
class NativeRead(pllm.Workload):
    """Control: every layer reads all positions and stores all KV as usual."""

    @pllm.custom_attention()
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        return kv.attend(q)[0]


@dataclass(frozen=True)
class QuestRead(pllm.Workload):
    pages: int = 8          # Pages read per head (16 tokens per page)
    recent_only: bool = False  # Control: same budget, reading only recent positions
    depth: int = 24         # Model layers: 24 for 0.5B; pass 28 for the 7B throughput run at the end of Step 3

    def later_layers(self) -> tuple[int, ...]:
        return tuple(range(2, self.depth))

    @pllm.custom_attention(layers=later_layers)
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        block = kv.key_cache.shape[1]
        if self.recent_only:
            return kv.attend(q, window=self.pages * block)[0]
        scores = page_scores(q, kv)
        page = torch.arange(scores.shape[-1], device=q.device)
        scores = scores.masked_fill(page * block > kv.positions[:, None, None], -float("inf"))  # Unwritten pages
        chosen = scores.topk(min(self.pages, scores.shape[-1]), dim=-1).indices            # [rows, heads, pages]
        tokens = (chosen[..., None] * block + torch.arange(block, device=q.device)).flatten(2)
        return kv.attend(q, topk=tokens)[0]  # Each head reads only its selected positions

In [6]:
for workload in (NativeRead(), QuestRead()):
    for line in str(pllm.explain(workload, CONFIG)).splitlines():
        if line.startswith("attention layer 2:"):
            print(f"{type(workload).__name__:10s}", line)
print(testing.check(QuestRead(), CONFIG))

NativeRead attention layer 2: custom `read`; all positions natively; 7340032 FLOP/token, <= 1048576 KV bytes/token [estimate at max_model_len; per-head budget upper bound]
QuestRead  attention layer 2: custom `read`; topk <= 128/head; 458752 FLOP/token, <= 458752 KV bytes/token [estimate at max_model_len; per-head budget upper bound]
status  contract / reason
PASS    tensor_parallel_size=1: admission
PASS    compile
PASS    custom-attention method math / CPU paged KV schedules
PASS    row permutation
PASS    request isolation
PASS    mixed prefill/decode
PASS    chunking / tokens=
PASS    preempted
SKIP    publish_lag: no row method reads versioned state
SKIP    neutral: pass neutral= with the author's disabled settings


In [7]:
for workload in (QuestRead(pages=8), QuestRead(pages=8, recent_only=True), QuestRead(pages=2)):
    with pllm.make_llm(CONFIG, workload=workload) as llm:
        print(f"{workload}: {answer(llm)!r}")

QuestRead(pages=8, recent_only=False, depth=24): '7361'
QuestRead(pages=8, recent_only=True, depth=24): ''
QuestRead(pages=2, recent_only=False, depth=24): '9'


First, examine the CPU estimates from `explain` (computed at `max_model_len=2048`):
- When `NativeRead` reads all positions, layer 2 reads up to `1048576` KV bytes/token at `7340032` FLOP/token;
- When `QuestRead` reads 128 positions per head (`topk <= 128/head`), compute drops to `458752` FLOP/token, yet each token still reads up to `458752` KV bytes/token.

Why is the KV read upper bound ~44% (`458752 / 1048576 ≈ 0.44`) when 128 positions is only 1/16 of 2048? Because this model's 14 query heads share 2 KV heads; when each query head chooses its own positions, the same KV head's data can be read separately by multiple query heads. `explain` surfaces that cost right at authoring time.

Next, `testing.check` on CPU: the custom attention logic remains invariant across paged KV scheduling—`row permutation`, `request isolation`, `mixed prefill/decode`, `chunking / tokens=`, and `preempted` all `PASS`.

Now look at the GPU answers:
- `QuestRead(pages=8, recent_only=False)` picking 8 pages (128 tokens) per head still answers `'7361'`;
- `QuestRead(pages=8, recent_only=True)` under the same 128-token budget outputs `''` (the first generated token is EOS) because the secret-code sentence lies outside the most recent 128 positions;
- `QuestRead(pages=2, recent_only=False)` cutting the budget to 2 pages per head answers `'9'`.

Note that this is a functional demo on a single prompt, not a quality benchmark. Also remember: read-only top-K reduces read traffic, not KV VRAM.

Now for Step 3: **If we want to actually reduce per-request KV VRAM, how do we declare KV storage layouts? How much VRAM does each layout save, and how does it affect model output?**

Below we compare four workloads whose method body is simply `return kv.attend(q)[0]`, differing only in the storage layout declared on `@pllm.custom_attention` (window trimming and sharing are handled by the storage layout itself):
1. `NativeRead`: keeps all KV by default.
2. `LocalGlobal`: even layers declare `keep="window", window=256`, retaining only the most recent 256 tokens of KV while older blocks are reclaimed automatically by vLLM's sliding-window mechanism; odd layers keep all KV.
3. `PromptPlusWindow`: even layers add `sink=True` to `keep="window", window=256`, retaining the entire prompt plus the most recent 256 generated tokens (corresponding to vLLM's R-SWA); odd layers stay unchanged.
4. `CrossLayerShared`: odd layers declare `keep="none", source=1`, allocating no KV of their own and reading directly from the layer 1 below (i.e., the preceding even layer).

Without starting the GPU engine, `pllm.explain` computes each request's maximum KV VRAM on CPU (an upper bound matching vLLM's own allocation cap, including tokens currently in flight).

In [8]:
@dataclass(frozen=True)
class LocalGlobal(pllm.Workload):
    """Even layers keep only the most recent 256 tokens of KV (sliding window); odd layers unchanged."""

    depth: int = 24  # Model depth (24 for 0.5B, 28 for 7B)

    def even_layers(self) -> tuple[int, ...]:
        return tuple(range(0, self.depth, 2))

    @pllm.custom_attention(layers=even_layers, keep="window", window=256)
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        return kv.attend(q)[0]


@dataclass(frozen=True)
class PromptPlusWindow(pllm.Workload):
    """Even layers keep the entire prompt plus the most recent 256 generated tokens."""

    depth: int = 24

    def even_layers(self) -> tuple[int, ...]:
        return tuple(range(0, self.depth, 2))

    @pllm.custom_attention(layers=even_layers, keep="window", window=256, sink=True)
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        return kv.attend(q)[0]


@dataclass(frozen=True)
class CrossLayerShared(pllm.Workload):
    """Odd layers store no KV and read directly from the layer below (CLA-style cross-layer sharing)."""

    depth: int = 24

    def odd_layers(self) -> tuple[int, ...]:
        return tuple(range(1, self.depth, 2))

    @pllm.custom_attention(layers=odd_layers, keep="none", source=1)
    def read(self, q: torch.Tensor, kv: pllm.model.PagedKV) -> torch.Tensor:
        return kv.attend(q)[0]


LAYOUTS = (NativeRead(), LocalGlobal(), PromptPlusWindow(), CrossLayerShared())
for workload in LAYOUTS:
    per_request = sum(capacity.bytes for capacity in pllm.explain(workload, CONFIG).kv_capacity)
    print(f"{type(workload).__name__:17s}: 每个请求最多 {per_request / 2**20:5.1f} MiB KV")

NativeRead       : 每个请求最多  24.0 MiB KV
LocalGlobal      : 每个请求最多  16.6 MiB KV
PromptPlusWindow : 每个请求最多  24.0 MiB KV
CrossLayerShared : 每个请求最多  12.0 MiB KV


In [9]:
for workload in LAYOUTS:
    with pllm.make_llm(CONFIG, workload=workload) as llm:
        concurrency = llm.diagnostics().kv_cache_max_concurrency  # Computed by vLLM from the actually allocated KV pool
        print(f"{type(workload).__name__:17s}: 可同时容纳 {concurrency:6.1f} 个 2048-token 请求；"
              f"回答 {answer(llm)!r}")

NativeRead       : 可同时容纳  358.1 个 2048-token 请求；回答 '7361'
LocalGlobal      : 可同时容纳  517.6 个 2048-token 请求；回答 '3'
PromptPlusWindow : 可同时容纳  358.1 个 2048-token 请求；回答 '7361'
CrossLayerShared : 可同时容纳  710.4 个 2048-token 请求；回答 'arga textDecorationugi-ingenSalesproj:oランキング étapelessKey'


Compare `explain`'s CPU estimates side by side with vLLM's actual pool allocation after startup:
- `explain` computes per-request max KV VRAM for the four workloads as: `NativeRead` `24.0 MiB`, `LocalGlobal` `16.6 MiB`, `PromptPlusWindow` `24.0 MiB`, `CrossLayerShared` `12.0 MiB`.
- After engine startup, `llm.diagnostics().kv_cache_max_concurrency` reports concurrency capacity from the actual allocated KV pool—how many 2048-token requests fit concurrently: `358.1`, `517.6`, `358.1`, and `710.4`.
- `LocalGlobal` reaches ~`1.45x` `NativeRead`'s concurrency, matching `24.0 / 16.6 ≈ 1.45`; `CrossLayerShared` reaches ~`1.98x`, matching `24.0 / 12.0 = 2.00`. Actual KV pool sizing also depends on compiler and workspace reservations, so concurrency does not scale purely from the per-request upper bound alone.
- `PromptPlusWindow`'s per-request upper bound stays `24.0 MiB`: in the worst case, a prompt fills all 2048 tokens and `sink=True` retains the whole prompt. It saves KV when post-prompt generation exceeds the window; actual concurrency vs. native can still differ slightly with workspace sizing.

Look at the answers on the secret-code question:
- `LocalGlobal` answers `'3'`, and `CrossLayerShared` emits gibberish `'arga textDecorationugi-ingenSalesproj:oランキング étapelessKey'`. As noted upfront, Qwen2.5-0.5B was not trained for local sliding windows or cross-layer KV sharing, so imposing those layouts breaks its answers.
- `PromptPlusWindow` still answers `'7361'`: generation is only 12 tokens and the full prompt is retained, so no KV is actually dropped.

Having inspected memory capacity, let's measure **the batched concurrent generation throughput of these custom reads and KV layouts**. 0.5B sufficed for inspecting behavior; for throughput we switch to `Qwen/Qwen2.5-7B-Instruct`: a 0.5B decode step takes only a few milliseconds, magnifying any fixed overhead, whereas 7B reflects typical deployment scale (bf16 weights ~15 GiB, requiring a 24 GiB GPU). Below we first compute the 7B per-request KV caps with `explain`, then benchmark throughput across 128 concurrent requests (8 prompts × 16 samples × 256 generated tokens) using `pllm.benchmark.throughput`: 1 warmup round followed by the median of 3 timed rounds, timing the full `generate` call (including prefill).

In [10]:
many = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=16, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)
# 7B has 28 layers, so declare layouts for 28 layers. Benchmark sequences are ~270 tokens, so cap context at 512 with up to 128 tokens/step:
# NativeRead, Quest, and CrossLayerShared use Inductor; window layouts preserve source ops. Memory utilization is 0.70 across all runs.
# QuestRead's PyTorch scorer materializes all pages' keys in FP32 per scheduled token; 7B's wider KV heads add >1 GiB of temporary VRAM.
BIG = pllm.ModelConfig("Qwen/Qwen2.5-7B-Instruct", max_model_len=512, max_num_seqs=128, max_num_batched_tokens=128,
                       enable_chunked_prefill=True, gpu_memory_utilization=0.70, seed=0,
                       execution=pllm.ExecutionOptions(cuda_graph_pool_budget_bytes=1 << 30))
for workload in (NativeRead(), LocalGlobal(depth=28), PromptPlusWindow(depth=28), CrossLayerShared(depth=28)):
    per_request = sum(capacity.bytes for capacity in pllm.explain(workload, replace(BIG, max_model_len=2048)).kv_capacity)
    print(f"7B {type(workload).__name__:17s}: 每个 2048-token 请求最多 {per_request / 2**20:5.1f} MiB KV")

rates = {}
with pllm.make_llm(BIG) as plain:
    rates["原生 vLLM"] = throughput(plain, many, sampling)
for workload in (NativeRead(), QuestRead(pages=8, depth=28), LocalGlobal(depth=28), CrossLayerShared(depth=28)):
    with pllm.make_llm(BIG, workload=workload) as llm:
        rates[type(workload).__name__] = throughput(llm, many, sampling)
for name, rate in rates.items():
    print(f"7B {name:17s}: {rate:6.0f} token/s，为原生 vLLM 的 {rate / rates['原生 vLLM']:.0%}")


7B NativeRead       : 每个 2048-token 请求最多 112.0 MiB KV
7B LocalGlobal      : 每个 2048-token 请求最多  70.4 MiB KV
7B PromptPlusWindow : 每个 2048-token 请求最多 112.0 MiB KV
7B CrossLayerShared : 每个 2048-token 请求最多  56.0 MiB KV
7B 原生 vLLM          :   4855 token/s，为原生 vLLM 的 100%
7B NativeRead       :   4817 token/s，为原生 vLLM 的 99%
7B QuestRead        :    649 token/s，为原生 vLLM 的 13%
7B LocalGlobal      :   4804 token/s，为原生 vLLM 的 99%
7B CrossLayerShared :   4798 token/s，为原生 vLLM 的 99%


First, the 7B KV accounting (`explain` max KV VRAM per 2048-token request): `NativeRead` `112.0 MiB`, `LocalGlobal` drops to `70.4 MiB`, `PromptPlusWindow` stays `112.0 MiB`, and `CrossLayerShared` drops to `56.0 MiB`. 7B uses 4x the KV per layer per token of 0.5B (4 KV heads × head_dim 128 vs. 2 KV heads × head_dim 64) and has more layers, so the absolute savings are much larger at the same relative ratios.

Now look at throughput (RTX 4090, vLLM 0.32.0, Qwen2.5-7B-Instruct bf16; 8 prompts × n=16, 256 tokens each; `max_model_len=512`, max 128 tokens/step, `gpu_memory_utilization=0.70`; 1 warmup + median of 3 timed rounds, full `generate`): all runs have CUDA graphs enabled; `NativeRead`, `QuestRead`, and `CrossLayerShared` use Inductor, while `LocalGlobal` uses the eager compiler.
- Vanilla vLLM: `4855 token/s`.
- `NativeRead`: `4817 token/s`, `99%` of vanilla.
- `LocalGlobal`: `4804 token/s` (`99%`); `CrossLayerShared`: `4798 token/s` (`99%`). Total sequence length is ~270 tokens, so a 256-token window trims very little history; the KV accounting above shows the capacity savings directly.
- `QuestRead`: `649 token/s` (`13%`). Here `page_scores` reads all pages' keys in PyTorch every step as a didactic implementation—do not take this number as a claim about the Quest algorithm's intrinsic speed.

Configurations are timed sequentially, so ratios include graph structure, fused kernels, and run-to-run variance rather than representing an interleaved paired speedup study. Under Inductor, Quest's temporaries OOMed at the earlier 0.75 utilization, so all runs in this cell share 0.70 to leave room for temporary tensors.

Finally, we return to the initial monitoring workload `RecentMass` and run a full GPU contract check with `testing.check_on_gpu`. Here we pass `neutral=RecentMass()` itself: because a monitoring workload is designed never to alter model generation, it serves directly as its own no-op control against vanilla vLLM's full-vocabulary distribution. Inductor may select different fused kernels for different model graph structures, in which case exact bitwise token parity is not required: `check_on_gpu` teacher-forces vanilla's greedy and fixed-seed continuations and compares the full-vocabulary distribution, requiring bidirectional KL mean/P99 `<= 0.001/0.01` nats and TV mean/P99 `<= 0.01/0.05`. Paths using identical kernels still compare token IDs bitwise.

In [11]:
print(testing.check_on_gpu(RecentMass(), CONFIG, neutral=RecentMass()))

status  contract / reason
PASS    explain/probe
PASS    host syncs: eight steady decode steps
PASS    preempted: fixed per-token inputs, token IDs and terminal request state
PASS    neutral: teacher-forced full-vocabulary KL/TV vs vanilla; T=0: KL mean/P99 0/0, TV 0/0; T=0.8: KL mean/P99 0/0, TV 0/0


All four GPU contract checks `PASS`:
- `explain/probe` passes;
- `host syncs: eight steady decode steps`: no GPU-to-CPU synchronization occurs over 8 steady decode steps;
- `preempted`: after forced request preemption and recomputation, per-token inputs, generated token IDs, and terminal request state reproduce identically;
- `neutral: teacher-forced full-vocabulary KL/TV vs vanilla`: compares full-vocabulary distributions on vanilla's greedy and fixed-seed continuations, reporting mean and P99 against the thresholds above.

### Summary

- `@pllm.custom_attention` replaces only the attention read at the specified layers; native q/k/v projections and KV writes proceed as usual.
- `kv.attend` returns the native kernel's output and float32 LSE, supporting local windows via `window=` or per-head logical position selection via `topk=`; the method can also return an `aux` tensor readable downstream via `pllm.attention(layer, "aux")` (or `"lse"` for the LSE of that layer's first `kv.attend`).
- KV storage layouts (`keep="window"`, `window=`, `sink=True`, `keep="none"`, `source=`) are declared directly in the decorator arguments, delegating underlying allocation and reclamation to vLLM; `pllm.explain` reports per-request KV VRAM caps on CPU before starting the engine.
- Know the boundary: read-only top-K (selective reading) saves read bandwidth, not KV VRAM; score-based dynamic eviction freeing KV VRAM (such as H2O or SnapKV) is unsupported in pLLM.
- Constraints: currently requires single-GPU (`tensor_parallel_size=1`), FlashAttention 2 backend with unquantized KV, and prefix caching disabled; static layouts cannot be combined with `PositionState` or `@pllm.custom_forward`.

### Next

In the next guide, [11 Lookup Memory](11_lookup_memory.ipynb), we look at how to look up a large table by token ID and explicitly choose whether it lives in GPU VRAM or host memory.